## Importing Packages

In [5]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub
import os

## DagsHub MLflow Setup

In [7]:
dagshub.init(
    repo_owner='rishikumarpk', 
    repo_name='Mlflow-101', 
    mlflow=True
)

Initialized MLflow to track repo "rishikumarpk/Mlflow-101"

Repository rishikumarpk/Mlflow-101 initialized!

In [9]:
mlflow.set_experiment(
    "Boston Housing Regression PBLM 1"
)

<Experiment: artifact_location='mlflow-artifacts:/8015634ae75840d59ee83ca01a396780', creation_time=1786071064969, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786071064969, lifecycle_stage='active', name='Boston Housing Regression PBLM 1', tags={}, trace_location=None, workspace='default'>

## Data Loading and Processing\n\nThe classic Boston Housing dataset was removed from `sklearn.datasets` (`load_boston`) starting in scikit-learn 1.2 due to ethical concerns about one of its features. We load the same dataset from a public CSV mirror instead. The target column `medv` is the median value of owner-occupied homes (in $1000s).

In [10]:
url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url)

X = df.drop(columns=["medv"])
y = df["medv"]

print("Shape:", X.shape)
print(df.describe())

Shape: (506, 13)
             crim          zn       indus        chas         nox          rm  \
count  506.000000  506.000000  506.000000  506.000000  506.000000  506.000000   
mean     3.613524   11.363636   11.136779    0.069170    0.554695    6.284634   
std      8.601545   23.322453    6.860353    0.253994    0.115878    0.702617   
min      0.006320    0.000000    0.460000    0.000000    0.385000    3.561000   
25%      0.082045    0.000000    5.190000    0.000000    0.449000    5.885500   
50%      0.256510    0.000000    9.690000    0.000000    0.538000    6.208500   
75%      3.677083   12.500000   18.100000    0.000000    0.624000    6.623500   
max     88.976200  100.000000   27.740000    1.000000    0.871000    8.780000   

              age         dis         rad         tax     ptratio           b  \
count  506.000000  506.000000  506.000000  506.000000  506.000000  506.000000   
mean    68.574901    3.795043    9.549407  408.237154   18.455534  356.674032   
std     28

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

## Build Models

In [12]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        X_train,
        y_train
    ),
    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train
    ),
    (
        "XGBoost",
        XGBRegressor(
            random_state=42
        ),
        X_train,
        y_train
    ),
    (
        "XGBoost (Tuned)",
        XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            random_state=42
        ),
        X_train,
        y_train
    )
]

In [13]:
reports = []
trained_models = []
for model_name, model, X_tr, y_tr in models:
    model.fit(
        X_tr,
        y_tr
    )
    predictions = model.predict(
        X_test
    )

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    report = {
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }
    reports.append(report)

    trained_models.append(
        model
    )
    print("="*50)
    print(model_name)
    print("="*50)

    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R2:   {r2:.4f}")

Linear Regression
RMSE: 4.6387
MAE:  3.1627
R2:   0.7112
Random Forest
RMSE: 3.3339
MAE:  2.2713
R2:   0.8508
XGBoost
RMSE: 3.0768
MAE:  2.1110
R2:   0.8730
XGBoost (Tuned)
RMSE: 2.9607
MAE:  2.0588
R2:   0.8824


## Log All Experiments to DagsHub

In [14]:
for i, (model_name, model, _, _) in enumerate(models):
    report = reports[i]

    with mlflow.start_run(
        run_name=model_name
    ):

        mlflow.log_param(
            "Model",
            model_name
        )

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "RMSE",
            report["rmse"]
        )

        mlflow.log_metric(
            "MAE",
            report["mae"]
        )

        mlflow.log_metric(
            "R2",
            report["r2"]
        )

        if "XGBoost" in model_name:

            mlflow.xgboost.log_model(
                model,
                "model"
            )

        else:

            mlflow.sklearn.log_model(
                model,
                "model"
            )

print("Experiments logged!")

2026/08/07 08:21:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0/runs/299779c6bb04449bb386c3683132478a
🧪 View experiment at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0


2026/08/07 08:21:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0/runs/c3120543ad4b4edcbb35522e26780591
🧪 View experiment at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0


2026/08/07 08:22:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0/runs/e054f449aa2a479f877d25c1fbe8a9f2
🧪 View experiment at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0


2026/08/07 08:22:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost (Tuned) at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0/runs/fea97da461c4420396f09f8a2350de43
🧪 View experiment at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0
Experiments logged!


## Best Model and Reg to DH

In [15]:
best_index = np.argmax(
    [
        r["r2"]
        for r in reports
    ]
)
best_model_name = models[best_index][0]

best_model = trained_models[best_index]

best_report = reports[best_index]

print(
    "Best Model:",
    best_model_name
)

Best Model: XGBoost (Tuned)


In [16]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:
    mlflow.log_param(
        "Model",
        best_model_name
    )
    mlflow.log_metric(
        "RMSE",
        best_report["rmse"]
    )
    mlflow.log_metric(
        "R2",
        best_report["r2"]
    )
    if "XGBoost" in best_model_name:

        mlflow.xgboost.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Best_Model"
        )
    else:

        mlflow.sklearn.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Best_Model"
        )
    run_id = run.info.run_id


print(run_id)

2026/08/07 08:23:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Boston_Housing_Best_Model'.
2026/08/07 08:23:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Boston_Housing_Best_Model, version 1
Created version '1' of model 'Boston_Housing_Best_Model'.


🏃 View run Champion_XGBoost (Tuned) at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0/runs/eead7cf8a5c346c6baee97a264835b17
🧪 View experiment at: https://dagshub.com/rishikumarpk/Mlflow-101.mlflow/#/experiments/0
eead7cf8a5c346c6baee97a264835b17
